# 01 · Fine-tune ProtBERT-BFD into AMP-BERT

Fine-tunes the pre-trained **ProtBERT-BFD** language model on the Veltri training
set; the resulting model **is** AMP-BERT. Self-contained — every parameter lives in
the **Config** cell, so you never need to edit external files.

> Set **Runtime → Change runtime type → GPU** before running.

In [ ]:
# === Setup: install a transformers version with the classic BERT tokenizer ===
# (Colab's bundled transformers is too new and mishandles ProtBERT-BFD.)
%pip install -q transformers==4.46.3 "tokenizers>=0.20,<0.21" sentencepiece

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)   # expect 'cuda' — set Runtime > Change runtime type > GPU

## Config — edit anything here

In [ ]:
MODEL_NAME   = 'Rostlab/prot_bert_bfd'
MAX_LEN      = 200
NUM_EPOCHS   = 15      # set to 1 for a quick smoke test
LR           = 5e-5
BATCH_SIZE   = 1
GRAD_ACCUM   = 64      # effective batch size = BATCH_SIZE * GRAD_ACCUM
WEIGHT_DECAY = 0.1
FP16         = True
SEED         = 0

TRAIN_URL = 'https://raw.githubusercontent.com/BioGavin/AMP-BERT/main/data/raw/all_veltri.csv'
# Where to save AMP-BERT. To survive a Colab disconnect, mount Google Drive
# and point this at e.g. '/content/drive/MyDrive/amp_bert'.
MODEL_DIR = 'models/amp_bert'
OUTPUT_DIR = 'results/amp_bert_train'

In [ ]:
# === Core logic (self-contained; mirrors src/amp_bert/) ===
import pandas as pd, numpy as np
from torch.utils.data import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          Trainer, TrainingArguments)
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             matthews_corrcoef, roc_auc_score)


def load_dataset(path, shuffle=False, random_state=0):
    df = pd.read_csv(path, index_col=0)
    return df.sample(frac=1, random_state=random_state) if shuffle else df


class AmpDataset(Dataset):
    """Tokenised view over an AMP DataFrame (columns: aa_seq, AMP)."""
    def __init__(self, df, tokenizer_name=MODEL_NAME, max_len=MAX_LEN):
        self.df = df.reset_index(drop=True)
        self.tokenizer = AutoTokenizer.from_pretrained(
            tokenizer_name, do_lower_case=False, use_fast=False)
        self.max_len = max_len
        self.seqs = list(self.df['aa_seq'])
        self.labels = list(self.df['AMP'].astype(int))

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        seq = ' '.join(''.join(self.seqs[idx].split()))
        ids = self.tokenizer(seq, truncation=True, padding='max_length',
                             max_length=self.max_len)
        sample = {k: torch.tensor(v) for k, v in ids.items()}
        sample['labels'] = torch.tensor(self.labels[idx])
        return sample


def compute_metrics(pred):
    labels = pred.label_ids
    logits = pred.predictions
    preds = logits.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary', zero_division=0)
    m = {'accuracy': accuracy_score(labels, preds), 'f1': f1,
         'precision': precision, 'recall': recall,
         'mcc': matthews_corrcoef(labels, preds)}
    if len(np.unique(labels)) == 2:
        z = logits - logits.max(axis=-1, keepdims=True)
        probs = np.exp(z) / np.exp(z).sum(axis=-1, keepdims=True)
        m['roc_auc'] = roc_auc_score(labels, probs[:, 1])
    return m


def model_init():   # must be zero-arg: Trainer passes `trial` to 1-arg fns
    return AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)


def make_training_args(output_dir):
    return TrainingArguments(
        output_dir=output_dir, num_train_epochs=NUM_EPOCHS,
        learning_rate=LR, per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM, weight_decay=WEIGHT_DECAY,
        warmup_steps=0, logging_steps=100, eval_strategy='no',
        save_strategy='no', fp16=FP16, seed=SEED, report_to='none')

## Load data

In [ ]:
df = load_dataset(TRAIN_URL, shuffle=True, random_state=SEED)
print(df.shape)
df.head()

In [ ]:
train_dataset = AmpDataset(df)
len(train_dataset)

## Train

Full config (15 epochs) takes hours on one GPU. Start with `NUM_EPOCHS = 1` to
verify the pipeline end-to-end.

In [ ]:
trainer = Trainer(model_init=model_init, args=make_training_args(OUTPUT_DIR),
                  train_dataset=train_dataset, compute_metrics=compute_metrics)
trainer.train()

## Sanity check on training data, then save AMP-BERT

In [ ]:
_, _, metrics = trainer.predict(train_dataset)
print(metrics)

In [ ]:
trainer.save_model(MODEL_DIR)
print('saved to', MODEL_DIR)